# Ingestão Bronze

Lê os Parquet já salvos no Volume e cria as tabelas Delta **já governadas**: cada tabela nasce
com descrição, propriedades, tags e comentário em cada coluna, além dos metadados de
rastreabilidade (`_ingestion_timestamp`, `_ingestion_file`) que a camada Bronze exige.

Os comentários de coluna vêm de fontes oficiais sempre que existem: as 408 colunas do Censo
Escolar são descritas pelo dicionário de variáveis do INEP, carregado primeiro por isso, e as
123 do IDEB seguem o padrão de nomes documentado nas planilhas do INEP.

Nenhuma linha é filtrada e nenhuma coluna é renomeada: a Bronze permanece fiel à origem, e as
regras de negócio ficam na Silver.

In [ ]:
from pyspark.sql import functions as F

import os
import sys

# Funções compartilhadas entre os notebooks ficam em pipeline_utils.py, na mesma pasta.
_pasta = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
sys.path.insert(0, _pasta if _pasta.startswith("/Workspace") else f"/Workspace{_pasta}")
from pipeline_utils import criar_tabela  # noqa: E402

# Parâmetros: usados pelo Job (Jobs & Pipelines) e com padrão para execução interativa.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("landing", "/Volumes/workspace/raw/files")

CATALOG = dbutils.widgets.get("catalog")
LANDING = dbutils.widgets.get("landing")

spark.sql(f"USE CATALOG {CATALOG}")

In [ ]:
import re

# O dicionário vem primeiro: é ele que descreve as colunas do Censo Escolar.
FONTES = {
    "dicionario_censo_escolar": "dicionario_censo_escolar.parquet",
    "municipios_ibge": "municipios_ibge.parquet",
    "populacao_municipios": "populacao_municipios.parquet",
    "bolsa_familia_municipio": "bolsa_familia_brasil.parquet",
    "ideb_municipios": "ideb_municipios.parquet",
    "censo_escolar_escolas": "censo_escolar_escolas.parquet",
}

METADADOS = {
    "dicionario_censo_escolar": dict(
        descricao="Dicionário oficial de variáveis dos Microdados do Censo Escolar 2023 (INEP): descrição, tipo, tamanho e domínio de cada coluna.",
        fonte="inep", dominio="educacao", grao="variavel", atualizacao="anual"),
    "municipios_ibge": dict(
        descricao="Municípios brasileiros com UF e região, da API de Localidades do IBGE.",
        fonte="ibge", dominio="territorio", grao="municipio", atualizacao="eventual"),
    "populacao_municipios": dict(
        descricao="População residente por município no Censo Demográfico 2022 (IBGE, SIDRA tabela 4714).",
        fonte="ibge", dominio="territorio", grao="municipio", atualizacao="decenal"),
    "bolsa_familia_municipio": dict(
        descricao="Novo Bolsa Família de 05/2024 agregado por município. A agregação, feita antes da ingestão, removeu nome, NIS e CPF dos beneficiários.",
        fonte="portal_transparencia", dominio="assistencia_social", grao="municipio_mes", atualizacao="mensal"),
    "ideb_municipios": dict(
        descricao="IDEB por município, rede e etapa de ensino, série histórica de 2005 a 2023 (INEP).",
        fonte="inep", dominio="educacao", grao="municipio_rede_etapa", atualizacao="bienal"),
    "censo_escolar_escolas": dict(
        descricao="Microdados do Censo Escolar 2023 (INEP): uma linha por escola, com identificação, infraestrutura e contagens de matrículas, docentes e turmas.",
        fonte="inep", dominio="educacao", grao="escola", atualizacao="anual"),
}

COMENTARIOS_FIXOS = {
    "_ingestion_timestamp": "Momento em que a linha foi carregada na camada Bronze.",
    "_ingestion_file": "Arquivo de origem na landing zone.",
    # dicionário
    "nome_variavel": "Nome da variável nos microdados do Censo Escolar.",
    "descricao": "Descrição oficial da variável, segundo o INEP.",
    "tipo": "Tipo declarado pelo INEP. Domínio: Num, Char.",
    "tamanho": "Tamanho máximo declarado pelo INEP.",
    "dominio_valores": "Valores aceitos pela variável, quando categórica (ex.: 0 - Não / 1 - Sim).",
    # IBGE e Bolsa Família
    "codigo_municipio_ibge": "Código IBGE do município, 7 dígitos.",
    "nome_municipio": "Nome do município.",
    "sigla_uf": "Sigla da unidade da federação.",
    "nome_regiao": "Região geográfica. Domínio: Norte, Nordeste, Centro-Oeste, Sudeste, Sul.",
    "ano": "Ano de referência.",
    "populacao": "População residente apurada no Censo Demográfico 2022.",
    "codigo_municipio_siafi": "Código SIAFI do município, usado pelo Portal da Transparência. Traduzido para IBGE na extração.",
    "mes_referencia": "Mês de competência dos pagamentos (AAAAMM).",
    "valor_total": "Soma das parcelas pagas no município no mês, em R$.",
    "quantidade_beneficiados": "Quantidade de benefícios pagos no município no mês.",
    # IDEB
    "CO_MUNICIPIO": "Código IBGE do município, 7 dígitos.",
    "NO_MUNICIPIO": "Nome do município.",
    "SG_UF": "Sigla da unidade da federação.",
    "REDE": "Rede de ensino. Domínio: Estadual, Municipal, Federal, Pública (consolidação feita pelo INEP).",
    "etapa_ensino": "Etapa do ensino fundamental, derivada do arquivo de origem. Domínio: Anos Iniciais, Anos Finais.",
}

# Nas planilhas do IDEB o mesmo sufixo muda de significado conforme a etapa:
# VL_APROVACAO_<ano>_1 é o 2º ano nos Anos Iniciais e o 6º ano nos Anos Finais.
PADROES_IDEB = [
    (r"VL_OBSERVADO_(\d{4})", "IDEB observado em {0}. Escala de 0 a 10."),
    (r"VL_PROJECAO_(\d{4})", "Meta de IDEB projetada pelo INEP para {0}."),
    (r"VL_NOTA_MATEMATICA_(\d{4})", "Proficiência média em Matemática no SAEB, {0}. Escala SAEB."),
    (r"VL_NOTA_PORTUGUES_(\d{4})", "Proficiência média em Língua Portuguesa no SAEB, {0}. Escala SAEB."),
    (r"VL_NOTA_MEDIA_(\d{4})", "Nota média padronizada do SAEB, {0}. Escala de 0 a 10."),
    (r"VL_INDICADOR_REND_(\d{4})", "Indicador de rendimento (fluxo escolar), {0}. Escala de 0 a 1."),
    (r"VL_APROVACAO_(\d{4})_SI_4", "Taxa de aprovação (%) da etapa inteira em {0}: 1º ao 5º ano (Anos Iniciais) ou 6º ao 9º ano (Anos Finais)."),
    (r"VL_APROVACAO_(\d{4})_SI", "Taxa de aprovação (%) do 1º ano em {0}. Existe só nos Anos Iniciais."),
    (r"VL_APROVACAO_(\d{4})_1", "Taxa de aprovação (%) em {0}: 2º ano (Anos Iniciais) ou 6º ano (Anos Finais)."),
    (r"VL_APROVACAO_(\d{4})_2", "Taxa de aprovação (%) em {0}: 3º ano (Anos Iniciais) ou 7º ano (Anos Finais)."),
    (r"VL_APROVACAO_(\d{4})_3", "Taxa de aprovação (%) em {0}: 4º ano (Anos Iniciais) ou 8º ano (Anos Finais)."),
    (r"VL_APROVACAO_(\d{4})_4", "Taxa de aprovação (%) em {0}: 5º ano (Anos Iniciais) ou 9º ano (Anos Finais)."),
]


def comentarios_para(tabela, colunas):
    comentarios = {c: COMENTARIOS_FIXOS[c] for c in colunas if c in COMENTARIOS_FIXOS}
    if tabela == "ideb_municipios":
        for coluna in colunas:
            for padrao, texto in PADROES_IDEB:
                achado = re.fullmatch(padrao, coluna)
                if achado:
                    comentarios[coluna] = texto.format(*achado.groups())
                    break
    if tabela == "censo_escolar_escolas":
        for linha in spark.table(f"{CATALOG}.bronze.dicionario_censo_escolar").collect():
            if linha["nome_variavel"] in colunas:
                texto = linha["descricao"]
                if linha["dominio_valores"]:
                    texto += f". Domínio: {linha['dominio_valores']}"
                comentarios[linha["nome_variavel"]] = texto
    return comentarios


for tabela, arquivo in FONTES.items():
    caminho = f"{LANDING}/{arquivo}"
    info = METADADOS[tabela]
    df = (
        spark.read.parquet(caminho)
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_ingestion_file", F.lit(caminho))
    )
    comentarios = comentarios_para(tabela, df.columns)
    criar_tabela(
        df,
        f"{CATALOG}.bronze.{tabela}",
        info["descricao"],
        propriedades={"camada": "bronze", "fonte": info["fonte"], "grao": info["grao"],
                      "atualizacao": info["atualizacao"], "arquivo_origem": arquivo},
        tags={"camada": "bronze", "fonte": info["fonte"], "dominio": info["dominio"],
              "dados_pessoais": "nao", "licenca": "dados_abertos"},
        comentarios=comentarios,
    )
    print(f"{tabela:28} {spark.table(f'{CATALOG}.bronze.{tabela}').count():>9,} linhas | "
          f"{len(comentarios):>3} de {len(df.columns)} colunas descritas")

## Validação: reconciliação com a origem

Cada tabela Delta é comparada com o Parquet de onde veio: a contagem tem de ser idêntica e
não pode ser zero.

In [0]:
for tabela, arquivo in FONTES.items():
    na_origem = spark.read.parquet(f"{LANDING}/{arquivo}").count()
    na_bronze = spark.table(f"{CATALOG}.bronze.{tabela}").count()
    assert na_bronze > 0, f"{tabela}: tabela vazia"
    assert na_bronze == na_origem, f"{tabela}: origem {na_origem:,} x bronze {na_bronze:,}"
    print(f"{tabela:28} {na_bronze:>9,} linhas = origem | status: OK")

In [0]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.bronze"))